### Imports + paths + load data (classification)


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Notebook location: ml/models/research/A5/Bootstrap/
SCORES_PATH   = Path("../../../../data/processed/20190108_scores_and_weak_links.csv")
FEATURES_PATH = Path("../../../../data/processed/AimoScore_WeakLink_big_scores.csv")

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

print("scores:", scores.shape)
print("big:", big.shape)

print(scores.head(2))
print(big.head(2))

import warnings
from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter("ignore", InsecureRequestWarning)



scores: (2096, 17)
big: (2094, 43)
                                            ID        Date     SCORE  \
0  cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect  2018-07-19  0.612532   
1  6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect  2018-07-30  0.388809   

   ForwardHead  LeftArmFallForward  RightArmFallForward  \
0         0.31                1.34                 1.15   
1         1.49                1.42                 1.39   

   LeftShoulderElevation  RightShoulderElevation  ExcessiveForwardLean  \
0                   0.33                    0.49                  0.00   
1                   0.00                    0.00                  1.08   

   LeftAsymmetricalWeightShift  RightAsymmetricalWeightShift  \
0                          0.0                          0.08   
1                          0.1                          0.00   

   LeftKneeMovesInward  RightKneeMovesInward  LeftKneeMovesOutward  \
0                  0.0                   0.0                   0.0   
1             

## Utilities + metrics (classification)

In [2]:
from typing import Dict, List
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss
)

def classification_metrics(y_true, y_pred, y_proba=None, classes=None) -> Dict[str, float]:
    m = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    if y_proba is not None and classes is not None:
        m["log_loss"] = float(log_loss(y_true, y_proba, labels=list(classes)))
    return m


def make_bootstrap_indices(n: int, seed: int) -> np.ndarray:
    """Bootstrap sample indices (with replacement), size n."""
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=n, size=n, endpoint=False)


def oob_mask(n: int, boot_idx: np.ndarray) -> np.ndarray:
    """Out-of-bag mask: True for samples NOT included in bootstrap draw."""
    mask = np.ones(n, dtype=bool)
    mask[np.unique(boot_idx)] = False
    return mask


## MLflow setup

### Locate env file

In [5]:
from pathlib import Path
from dotenv import load_dotenv
from os import getenv
import mlflow

# Path to project root (adjust depth if needed)
PROJECT_ROOT = Path("../../../../").resolve()

env_path = PROJECT_ROOT / "ml.env"
print("Loading .env from:", env_path)

load_dotenv(dotenv_path=env_path)

uri = getenv("MLFLOW_TRACKING_URI")
print("MLFLOW_TRACKING_URI =", uri)

if not uri:
    raise RuntimeError("MLFLOW_TRACKING_URI not found in .env file.")

mlflow.set_tracking_uri(uri)


Loading .env from: /Users/pettergustafsson/study/4DT907/ml_project/machine_learning_project_4dt907/ml/models/ml.env
MLFLOW_TRACKING_URI = https://172.27.62.170:5001/


In [7]:

experiment_name = "A5_bootstrap_classification"
mlflow.set_experiment(experiment_name)
print("Experiment:", experiment_name)


Experiment: A5_bootstrap_classification


## Build classification labels (WeakestLink) + merge features

In [8]:
# --- Classification: Predict WeakestLink ---

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

# Optional: drop some classes if your team previously did so
DROP_CLASSES = True
DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}
DROP_TIES = True

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

# Make sure indicators are numeric
scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

# Compute weakest link + ties
row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

scores_labeled = scores.loc[~scores["HasTie"]].copy() if DROP_TIES else scores.copy()
scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

labels = scores_labeled[["ID", "WeakestLink"]]

# Merge labels onto feature table
data = big.merge(labels, on="ID", how="inner")

# Feature selection (same pattern as your A4 notebook)
angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X_clf = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y_clf = data["WeakestLink"].astype(str)

# Keep only fully numeric rows (avoids NaN issues; you can relax this later if you want)
mask = X_clf.notna().all(axis=1)
X_clf = X_clf.loc[mask].reset_index(drop=True)
y_clf = y_clf.loc[mask].reset_index(drop=True)

all_classes = np.array(sorted(y_clf.unique()), dtype=object)

print("X_clf:", X_clf.shape, "y_clf:", y_clf.shape)
print("n_classes:", len(all_classes))
print("classes:", all_classes[:10], "..." if len(all_classes) > 10 else "")
print("class counts:\n", y_clf.value_counts().head(10))


X_clf: (1724, 38) y_clf: (1724,)
n_classes: 10
classes: ['ExcessiveForwardLean' 'ForwardHead' 'LeftArmFallForward'
 'LeftAsymmetricalWeightShift' 'LeftKneeMovesOutward'
 'LeftShoulderElevation' 'RightArmFallForward' 'RightKneeMovesInward'
 'RightKneeMovesOutward' 'RightShoulderElevation'] 
class counts:
 WeakestLink
RightArmFallForward            454
LeftArmFallForward             328
RightKneeMovesOutward          278
RightShoulderElevation         252
ExcessiveForwardLean            99
ForwardHead                     96
LeftAsymmetricalWeightShift     71
LeftShoulderElevation           51
LeftKneeMovesOutward            49
RightKneeMovesInward            46
Name: count, dtype: int64


## Train/test split (fixed) + bootstrap config

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_clf,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_clf
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

N_BOOTSTRAPS = 25
BOOTSTRAP_SEED0 = 20_000


Train: (1379, 38) (1379,)
Test : (345, 38) (345,)


## Define a baseline classifier (simple + stable)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_SEED
    ))
])

base_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

## Bootstrap loop + MLflow logging (classification)

In [17]:
from sklearn.base import clone
from pathlib import Path

ART_DIR = Path("./_mlflow_artifacts_bootstrap_clf")
ART_DIR.mkdir(parents=True, exist_ok=True)

bootstrap_test_metrics: List[Dict[str, float]] = []

with mlflow.start_run(run_name="bootstrap_parent") as parent_run:
    mlflow.log_params({
        "random_seed": RANDOM_SEED,
        "n_bootstraps": N_BOOTSTRAPS,
        "bootstrap_seed0": BOOTSTRAP_SEED0,
        "train_size": int(X_train.shape[0]),
        "test_size": int(X_test.shape[0]),
        "model_class": type(base_model.named_steps["clf"]).__name__,
        "drop_ties": DROP_TIES,
        "drop_classes": DROP_CLASSES,
        "n_classes": int(len(all_classes)),
    })
    
    mlflow.log_param("n_features", int(X_train.shape[1]))

    feat_path = ART_DIR / "feature_cols.txt"
    feat_path.write_text("\n".join(feature_cols), encoding="utf-8")
    mlflow.log_artifact(str(feat_path), artifact_path="data")


    for b in range(N_BOOTSTRAPS):
        seed = BOOTSTRAP_SEED0 + b
        n_train = X_train.shape[0]

        boot_idx = make_bootstrap_indices(n_train, seed=seed)
        Xb = X_train.iloc[boot_idx]
        yb = y_train.iloc[boot_idx]

        # OOB set (optional)
        mask_oob = oob_mask(n_train, boot_idx)
        X_oob = X_train.loc[mask_oob]
        y_oob = y_train.loc[mask_oob]

        m = clone(base_model)

        with mlflow.start_run(run_name=f"bootstrap_{b:03d}", nested=True):
            mlflow.log_params({
                "bootstrap_id": b,
                "bootstrap_seed": seed,
                "bootstrap_sample_size": int(len(boot_idx)),
                "bootstrap_unique_count": int(len(np.unique(boot_idx))),
                "oob_size": int(X_oob.shape[0]),
            })

            # Fit
            m.fit(Xb, yb)

            # Test predictions
            yhat_test = m.predict(X_test)

            # Probabilities (for log_loss)
            yproba_test = None
            if hasattr(m, "predict_proba"):
                yproba_test = m.predict_proba(X_test)

            classes_for_loss = getattr(m, "classes_", all_classes)
            test_m = classification_metrics(y_test, yhat_test, yproba_test, classes=classes_for_loss)

            for k, v in test_m.items():
                mlflow.log_metric(f"test_{k}", v)

            bootstrap_test_metrics.append(test_m)

            # OOB metrics (optional)
            if X_oob.shape[0] > 0:
                yhat_oob = m.predict(X_oob)
                yproba_oob = m.predict_proba(X_oob) if hasattr(m, "predict_proba") else None
                classes_for_loss = getattr(m, "classes_", all_classes)
                oob_m = classification_metrics(y_oob, yhat_oob, yproba_oob, classes=classes_for_loss)
                for k, v in oob_m.items():
                    mlflow.log_metric(f"oob_{k}", v)

            # Save bootstrap indices as artifact
            idx_path = ART_DIR / f"bootstrap_indices_{b:03d}.npy"
            np.save(idx_path, boot_idx)
            mlflow.log_artifact(str(idx_path), artifact_path="bootstrap_indices")

            # Log model
            mlflow.sklearn.log_model(m, name="model")

    # Parent-level summary
    def _mean_std(key: str):
        vals = [m[key] for m in bootstrap_test_metrics if key in m]
        return float(np.mean(vals)), float(np.std(vals))

    for key in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]:
        mean_v, std_v = _mean_std(key)
        mlflow.log_metric(f"bootstrap_test_{key}_mean", mean_v)
        mlflow.log_metric(f"bootstrap_test_{key}_std", std_v)

    # log_loss may exist too
    if "log_loss" in bootstrap_test_metrics[0]:
        mean_v, std_v = _mean_std("log_loss")
        mlflow.log_metric("bootstrap_test_log_loss_mean", mean_v)
        mlflow.log_metric("bootstrap_test_log_loss_std", std_v)

    print("Parent run:", parent_run.info.run_id)
    print("Bootstrap test accuracy mean/std:", *_mean_std("accuracy"))
    print("Bootstrap test weighted_f1 mean/std:", *_mean_std("weighted_f1"))


🏃 View run bootstrap_000 at: https://172.27.62.170:5001/#/experiments/3/runs/fdb79501ce464d50bade63e4ab9fa5c9
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/3
🏃 View run bootstrap_001 at: https://172.27.62.170:5001/#/experiments/3/runs/12389642b7f741c8b514f6b9e24bc039
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/3
🏃 View run bootstrap_002 at: https://172.27.62.170:5001/#/experiments/3/runs/2db5e8d8cd76472f9d66285d507b0442
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/3
🏃 View run bootstrap_003 at: https://172.27.62.170:5001/#/experiments/3/runs/ca11c0f2f3504105a56a8ae0ef3d7ec8
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/3
🏃 View run bootstrap_004 at: https://172.27.62.170:5001/#/experiments/3/runs/9379d713949e4fd6bd1542751241904f
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/3
🏃 View run bootstrap_005 at: https://172.27.62.170:5001/#/experiments/3/runs/c3b8482b61be47f7b78ec277130695c1
🧪 View experime

## Summary table

In [18]:
summary_df = pd.DataFrame(bootstrap_test_metrics)
print(summary_df.describe().T)

                   count      mean       std       min       25%       50%  \
accuracy            25.0  0.589101  0.014524  0.565217  0.573913  0.588406   
balanced_accuracy   25.0  0.475917  0.026694  0.428640  0.460980  0.470739   
macro_f1            25.0  0.464297  0.025953  0.420304  0.451542  0.461793   
weighted_f1         25.0  0.578684  0.015968  0.553378  0.561553  0.580942   
log_loss            25.0  1.153976  0.037966  1.088671  1.122236  1.159984   

                        75%       max  
accuracy           0.600000  0.608696  
balanced_accuracy  0.488440  0.547103  
macro_f1           0.479836  0.538693  
weighted_f1        0.591091  0.601052  
log_loss           1.186924  1.206082  
